001 hello world
002 specter flag, drop cat 1, swich 2/3/4 to low/mid/high

In [ ]:
from ppp_paths import dataraw, dataint, datafin, dataconf
writeFile = True

In [ ]:
import pandas as pd

# Assuming datafin is defined as './data/fin/'
datafin = './data/fin/'

# Read the CSV file
finalppps = pd.read_csv(datafin + 'finalppps.csv', dtype={'magid': str, 'patent_id': str, 'ppp_score': int})
# Display the first few rows to verify
finalppps
#.value_counts('ppp_score').sort_values(ascending=False)()

In [ ]:
finalpppsllmadjusted = finalppps
# finalpppsllmadjusted = finalppps[['magid', 'patent_id', 'ppp_score']]
finalpppsllmadjusted.value_counts('ppp_score').sort_values(ascending=False)

In [ ]:
import pandas as pd

# Load the first CSV file
llmadjustments1 = pd.read_csv(dataraw + 'ppp_grok4_refined_4_allDONOTDELETE.csv', dtype={'magid': str, 'patent_id': str})
print(len(llmadjustments1))
# Load the second CSV file
llmadjustments2 = pd.read_csv(dataraw + 'ppp_grok4_refined_4_allDONOTDELETE_MISSING.csv', dtype={'magid': str, 'patent_id': str})
print(len(llmadjustments2))
# Load the third CSV file
llmadjustments3 = pd.read_csv(dataraw + 'ppp_grok4_refined_3DONOTDELETE.csv', dtype={'magid': str, 'patent_id': str})
print(len(llmadjustments3))
# Combine them into a single DataFrame
llmadjustments = pd.concat([llmadjustments1, llmadjustments2, llmadjustments3], ignore_index=True)

# Display the first few rows to verify
print(llmadjustments.shape)
llmadjustments.value_counts('assessment').sort_values(ascending=False)

In [ ]:
import numpy as np

# Merge finalpppsllmadjusted with llmadjustments on magid and patent_id
finalpppsllmadjusted = finalpppsllmadjusted.merge(llmadjustments[['magid', 'patent_id', 'assessment']], on=['magid', 'patent_id'], how='left')


In [ ]:
# After the merge, calculate the number of ppp_score=4 rows without a match in llmadjustments
no_match_df = finalpppsllmadjusted[(finalpppsllmadjusted['ppp_score'] == 4) & (finalpppsllmadjusted['assessment'].isna())]

# Number without matches
num_no_matches = len(no_match_df)
print(f"Number of ppp_score=4 rows without matches in llmadjustments: {num_no_matches}")

# The df of just the ones without matches is no_match_df
no_match_df

In [ ]:

# Adjust ppp_score based on assessment
finalpppsllmadjusted['ppp_score'] = np.select(
    [
        finalpppsllmadjusted['assessment'] == 'A',
        # finalpppsllmadjusted['assessment'] == 'B',
        finalpppsllmadjusted['assessment'] == 'D',
        finalpppsllmadjusted['assessment'] == 'C'
    ],
    [
        5,
        # 5,
        1,
        2
    ],
    default=finalpppsllmadjusted['ppp_score']  # Keep original if not matching
)

# Drop the assessment column if no longer needed
finalpppsllmadjusted = finalpppsllmadjusted.drop(columns=['assessment'])

# Display value counts
finalpppsllmadjusted.value_counts('ppp_score').sort_values(ascending=False)

In [ ]:
finalpppsllmadjusted = finalpppsllmadjusted[finalpppsllmadjusted['ppp_score'] != 1].copy()
finalpppsllmadjusted['ppp_score'] = finalpppsllmadjusted['ppp_score'] - 1
finalpppsllmadjusted.value_counts('ppp_score').sort_values(ascending=False)


In [ ]:
readjust = True
if readjust is True:
    affected_rows = finalpppsllmadjusted[(finalpppsllmadjusted['ppp_score'] == 4) & (finalpppsllmadjusted['daysdiffnoabs'] < -180) & (finalpppsllmadjusted['authors in common'] < 3)].shape[0]
    print(f"Number of rows affected: {affected_rows}")
    finalpppsllmadjusted.loc[(finalpppsllmadjusted['ppp_score'] == 4) & (finalpppsllmadjusted['daysdiffnoabs'] < -180) & (finalpppsllmadjusted['authors in common'] < 3), 'ppp_score'] = 3

In [ ]:
# mask_ab = (finalpppsllmadjusted['ppp_score'] == 4) #& (categorized['response'].str.upper().isin(['C', 'D']))
# # ab_joined = (
# #     categorized.loc[mask_ab, ['magid', 'patent_id']]
# #     .merge(finalppps[['magid', 'patent_id', 'daysdiffnoabs', 'authors in common']], on=['magid', 'patent_id'], how='left')
# # )
# count_ab = mask_ab[(mask_ab['daysdiffnoabs'] < -180) & (mask_ab['authors in common'] < 3)].shape[0]
# print(f"Category 4 with daysdiffnoabs < -180 and authors in common < 3: {count_ab}")


In [ ]:
finalpppsllmadjusted = (
    finalpppsllmadjusted[['magid', 'patent_id', 'ppp_score']]
    .sort_values('ppp_score', ascending=False)
    .drop_duplicates(['magid', 'patent_id'], keep='first')
    .reset_index(drop=True)
)
finalpppsllmadjusted.value_counts('ppp_score').sort_values(ascending=False)


In [ ]:
finalpppsllmadjusted.columns

In [ ]:
finalpppsllmadjusted.to_csv(datafin + 'finalpppsadjusted_prescored.csv', index=False)

START HERE FOR RESCORE

In [ ]:
finalpppsllmadjusted2test = pd.read_csv(datafin + 'finalpppsadjusted_prescored.csv', dtype={'magid': str, 'patent_id': str, 'ppp_score': int})
# finalpppsllmadjusted2test = finalpppsllmadjusted2test.drop_duplicates
finalpppsllmadjusted2test

In [ ]:
# finalpppsllmadjusted2test['original_ppp_score'] = finalpppsllmadjusted2test['ppp_score']
# finalpppsllmadjusted2test

In [ ]:
import os
from glob import glob

handcheck_dir = os.path.join(dataconf, 'authorhandcheck')
csv_paths = glob(os.path.join(handcheck_dir, '*.csv'))
if not csv_paths:
    print(f'WARNING: no author-validation files in {handcheck_dir} (confidential, not publicly distributed). Skipping accuracy scoring; the dataset itself is unaffected.')

handchecked_frames = []
for path in csv_paths:
    print(f"Reading file: {path}")
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.replace('\ufeff', '', regex=False)
    df = df.rename(columns={col: col.lower() for col in df.columns})

    if 'patentid' in df.columns:
        df = df.rename(columns={'patentid': 'patent_id'})
    if 'magid' not in df.columns or 'patent_id' not in df.columns:
        print(f"Skipping {path}: missing magid or patent_id")
        continue

    if 'response' not in df.columns:
        print(f"Skipping {path}: no 'response' column after normalization")
        continue

    df['magid'] = df['magid'].astype(str)
    df['patent_id'] = df['patent_id'].astype(str)
    handchecked_frames.append(df)

if not handchecked_frames:
    handchecked_frames = [pd.DataFrame(columns=['magid', 'patent_id', 'response'])]

handchecked = pd.concat(handchecked_frames, ignore_index=True)
handchecked = handchecked[['magid', 'patent_id', 'response']]
handchecked = handchecked[~(
    (handchecked['magid'] == '1986017883') &
    (handchecked['patent_id'] == '8088587') &
    (handchecked['response'].str.upper() == 'D')
)]
categorized = handchecked[handchecked['response'].isin(['A', 'B', 'C', 'D'])]
categorized = categorized.drop_duplicates()

duplicates = categorized[categorized.duplicated(['magid', 'patent_id'], keep=False)]
print(f"Duplicate magid/patent_id rows: {len(duplicates)}")
if len(duplicates):
    print(duplicates)

print(f"Categorized rows: {len(categorized)}")
categorized.value_counts('response').sort_values(ascending=False)


In [ ]:
# affected_rows = finalppps[(finalppps['ppp_score'] == 4) & (finalppps['daysdiff'] < -180) & (finalppps['authors in common'] < 3)].shape[0]
# print(f"Number of rows affected: {affected_rows}")
# finalppps.loc[(finalppps['ppp_score'] == 4) & (finalppps['daysdiff'] < -180) & (finalppps['authors in common'] < 3), 'ppp_score'] = 3

In [ ]:
handchecked = handchecked.merge(finalpppsllmadjusted2test, on=['magid', 'patent_id'], how='inner')
categorized = handchecked[handchecked['response'].isin(['A', 'B', 'C', 'D'])]  # Re-filter after merge
# categorized = categorized.drop_duplicates(subset=['magid', 'patent_id', 'response', 'ppp_score'])
print(categorized.columns)
categorized = (
    categorized.sort_values('ppp_score', ascending=False).drop_duplicates(['magid', 'patent_id'], keep='first').reset_index(drop=True)
)

print(f"matched rows: {len(categorized)}")

cross_tab = pd.crosstab(categorized['response'], categorized['ppp_score']).sort_index(axis=1, ascending=False).T
print("\nTransposed cross-tabulation of hand-checked scores (A/B/C/D) by ppp_score (4/3/2/1):")
print(cross_tab)
print("\nPercentages for each ppp_score:")
for score in cross_tab.index:
    row = cross_tab.loc[score]
    total = row.sum()
    if total > 0:
        ab_count = row.get('A', 0) + row.get('B', 0)
        a_count = row.get('A', 0)
        cd_count = row.get('C', 0) + row.get('D', 0)
        bcd_count = row.get('B', 0) + row.get('C', 0) + row.get('D', 0)
        ab_percent = ab_count / total * 100
        a_percent = a_count / total * 100
        cd_percent = cd_count / total * 100
        bcd_percent = bcd_count / total * 100
        print(f"PPP Score {score}:   Total = {total}")
        print(f"  A/B = {ab_count} ({ab_percent:.2f}%), C/D = {cd_count} ({cd_percent:.2f}%)")
        print(f"  A = {a_count} ({a_percent:.2f}%), B/C/D = {bcd_count} ({bcd_percent:.2f}%)")
    else:
        print(f"PPP Score {score}: No data")

In [ ]:
# # ...existing code...
# plagiarism_path = dataint + "PPP_positive_training_setPLAGIARISMtighter.tsv"
# self_plagiarism = pd.read_csv(plagiarism_path, sep="\t", dtype={"magid": str, "patent_id": str})

# hand_cd = categorized[categorized["response"].str.upper().isin(["C", "D", "B", "A"])][["magid", "patent_id", "response", "ppp_score"]]

# required_cols = {"magid", "patent_id"}
# missing_cols = required_cols - set(self_plagiarism.columns)
# if missing_cols:
#     raise KeyError(f"Missing columns in self-plagiarism file: {missing_cols}")

# cd_plagiarism = hand_cd.merge(
#     self_plagiarism[["magid", "patent_id"]],
#     on=["magid", "patent_id"],
#     how="inner"
# )
# cd_plagiarism = cd_plagiarism.sort_values('response')

# if cd_plagiarism.empty:
#     print("No hand-labeled C/D pairs found in the self-plagiarism subset.")
# else:
#     print(cd_plagiarism[["magid", "patent_id", "response", "ppp_score"]])
# # ...existing code...

In [ ]:
# mask = (categorized['ppp_score'] == 4) & (categorized['response'].str.upper() == 'C')
# selected = categorized.loc[mask, ['magid', 'patent_id']].drop_duplicates()
# full_records = (
#     selected
#     .merge(finalppps, on=['magid', 'patent_id'], how='left', suffixes=('', '_orig'))
#     .merge(finalpppsllmadjusted2test, on=['magid', 'patent_id'], how='left', suffixes=('', '_adjusted'))
# )
# print(full_records)

In [ ]:
# duplicates = categorized[categorized.duplicated(['magid', 'patent_id'], keep=False)]
# print(f"Duplicate magid/patent_id rows: {len(duplicates)}")
# if len(duplicates):
#     for (mid, pid), grp in duplicates.groupby(['magid', 'patent_id']):
#         differing = {
#             col: grp[col].unique().tolist()
#             for col in grp.columns
#             if col not in ['magid', 'patent_id'] and grp[col].nunique() > 1
#         }
#         if differing:
#             print(f"magid={mid}, patent_id={pid} differing fields: {differing}")
#         else:
#             print(f"magid={mid}, patent_id={pid} duplicate rows identical")

In [ ]:
# import matplotlib.pyplot as plt

# mask_cd = (categorized['ppp_score'] == 4)# & (categorized['response'].str.upper().isin(['C', 'D']))
# daysdiff_series = (
#     categorized.loc[mask_cd, ['magid', 'patent_id']]
#     .merge(finalppps, on=['magid', 'patent_id'], how='left')
#     ['daysdiffnoabs']
#     .dropna()
# )

# stats = daysdiff_series.agg(['count', 'min', 'max', 'mean', 'std'])
# print(stats)

# plt.figure(figsize=(8, 4))
# daysdiff_series.hist(bins=30)
# plt.title('Daysdiff distribution for PPP score 4 with response C/D')
# plt.xlabel('daysdiffnoabs')
# plt.ylabel('Frequency')
# plt.tight_layout()
# plt.show()


# mask_cd = (categorized['ppp_score'] == 4) #& (categorized['response'].str.upper().isin(['C', 'D']))
# daysdiff_counts = (
#     categorized.loc[mask_cd, ['magid', 'patent_id']]
#     .merge(finalppps, on=['magid', 'patent_id'], how='left')['daysdiffnoabs']
#     .dropna()
#     .value_counts()
#     .sort_index()
# )
# pd.set_option('display.max_rows', None)
# print(daysdiff_counts.to_string())
# pd.reset_option('display.max_rows')

In [ ]:
# mask_ab = (categorized['ppp_score'] == 4) #& (categorized['response'].str.upper().isin(['C', 'D']))
# ab_joined = (
#     categorized.loc[mask_ab, ['magid', 'patent_id']]
#     .merge(finalppps[['magid', 'patent_id', 'daysdiffnoabs', 'authors in common']], on=['magid', 'patent_id'], how='left')
# )
# count_ab = ab_joined[(ab_joined['daysdiffnoabs'] < -180) & (ab_joined['authors in common'] < 4)].shape[0]
# print(f"count: {count_ab}")


In [ ]:
finalpppsllmadjusted2save = pd.read_csv(datafin + 'finalpppsadjusted_prescored.csv', dtype={'magid': str, 'patent_id': str, 'ppp_score': int})
# drop if ppp_score is 1 or 2 (2 = 'low' confidence, excluded from the final output)
finalpppsllmadjusted2save = finalpppsllmadjusted2save[~finalpppsllmadjusted2save['ppp_score'].isin([1, 2])]
# convert ppp_score 4 to "High", 3 to "Mid"
finalpppsllmadjusted2save['ppp_score'] = finalpppsllmadjusted2save['ppp_score'].map({4: 'high', 3: 'mid'})
finalpppsllmadjusted2save
# reame ppp_score to confidence_level
finalpppsllmadjusted2save = finalpppsllmadjusted2save.rename(columns={'ppp_score': 'confidence_level'})
# print the column names
print(finalpppsllmadjusted2save.columns)    
# print total number of rows
print(f"Total rows after adjustment: {len(finalpppsllmadjusted2save)}")
if writeFile is True:
    finalpppsllmadjusted2save.to_csv(datafin + 'finalpppsadjusted.csv', index=False)
finalpppsllmadjusted2save['confidence_level'].value_counts()